In [1]:
import random
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB

# ------------------------------------------------------------
# Random seeds
# Results are reproducible using a set seed.
# ------------------------------------------------------------
np.random.seed(42)
random.seed(42)

# ------------------------------------------------------------
# 1. Sets & basic parameters
# These numbers are the size of the problem, number of courses, rooms, exam slots, and students.
# ------------------------------------------------------------
numberCourses = 28
numberRooms = 8
numberSlots = 12
numberDays = 4
slotsPerDay = 3
numberStudents = 1000

# Lists of names for each set (CS_100, CS_110, ... / Room_101, ... / Slot_1, ...)
courses = [f"CS_{100 + i*10}" for i in range(numberCourses)]
rooms = [f"Room_{101 + i}" for i in range(numberRooms)]
timeSlots = [f"Slot_{t+1}" for t in range(numberSlots)]

# Maps each slot to its exam day: Slot_1-3 = Day 1, Slot_4-6 = Day 2, etc.
# Only needed if you add a per-day rule to the model.
dayOfSlot = {timeSlots[t]: t // slotsPerDay + 1 for t in range(numberSlots)}

# ------------------------------------------------------------
# 2. Course enrollments & room capacities
# Randomly assign each course 25–200 students.
# Rooms have fixed capacities of 30–220 seats.
# ------------------------------------------------------------
courseEnrollments = {c: int(np.random.randint(25, 201)) for c in courses}
capacityList = [30, 45, 60, 90, 120, 150, 180, 220]
roomCapacities = {rooms[i]: capacityList[i] for i in range(numberRooms)}

# ------------------------------------------------------------
# 3. Simulate student-course enrollments
# Creates a table showing which courses each student takes.
# A 1 means the student is enrolled in that course.
# Each course is assigned its exact enrollment number of students.
# ------------------------------------------------------------
studentID = [f"S_{1000 + i}" for i in range(numberStudents)]
enrollmentMatrix = np.zeros((numberStudents, numberCourses), dtype=int)

for j, c in enumerate(courses):
    chosen = np.random.choice(numberStudents, size=courseEnrollments[c], replace=False)
    enrollmentMatrix[chosen, j] = 1

# Turn the matrix into a labeled table (rows = students, columns = courses)
studentCourseDataFrame = pd.DataFrame(enrollmentMatrix, index=studentID, columns=courses)

# ------------------------------------------------------------
# 4. Course-to-course conflict parameter
# Counts how many students take each pair of courses.
# Each pair is counted only once to avoid duplicates.
# ------------------------------------------------------------
overlap_dict = {}
for i, c1 in enumerate(courses):
    for j, c2 in enumerate(courses):
        if i < j:
            shared_students = np.sum(studentCourseDataFrame[c1] & studentCourseDataFrame[c2])
            overlap_dict[(c1, c2)] = int(shared_students)

# ------------------------------------------------------------
# Quick printouts to confirm the data was built correctly.
# ------------------------------------------------------------
print("Courses:", numberCourses)
print("Rooms:", numberRooms)
print("Time slots:", numberSlots)

print("Largest course:", max(courseEnrollments.values()))
print("Largest room:", max(roomCapacities.values()))

# Count the course pairs that share at least one student
conflictPairs = 0
for pair in overlap_dict:
    if overlap_dict[pair] > 0:
        conflictPairs = conflictPairs + 1
print("Course pairs sharing students:", conflictPairs, "of", len(overlap_dict))

# ------------------------------------------------------------
# 5. Export data
# ------------------------------------------------------------

# Save student enrollment data
studentCourseDataFrame.to_csv("student_enrollments.csv")

# Save course conflict data
conflictDataFrame = pd.DataFrame(
    [(c1, c2, count) for (c1, c2), count in overlap_dict.items()],
    columns=["Course 1", "Course 2", "Shared Students"]
)

conflictDataFrame.to_csv("course_conflicts.csv", index=False)

print("CSV files exported successfully!")

Courses: 28
Rooms: 8
Time slots: 12
Largest course: 185
Largest room: 220
Course pairs sharing students: 376 of 378
CSV files exported successfully!
